# example how to generate a feature cube based on EO data (Sentinel-1 and Sentinel-2) for a 10x10km tile
this tests include the generation of the feature cube with and without NVBT band. The NVBT band is a measure of the number of valid input data timesteps after cloud masking and the internal temporal binning.
The NVBT band can be useful for identifying areas with high or low levels of observation, which can be important for a variety of applications, such as monitoring changes in land use or assessing the quality of data in a given area.

In [ ]:
from eo_processing.utils.helper import init_connection
from eo_processing.openeo.processing import generate_master_feature_cube
from eo_processing.config.settings import get_advanced_options, get_job_options, get_collection_options

#### declare space and time

In [ ]:
# the time context is given by start and end date
year = 2024
start = f'{year}-01-01'
end = f'{year+1}-01-01'   # the end is always exclusive

# the space context is defined as a bounding box dictionary with south,west,north,east and crs
# we take as example an 10x10km tile in EU LAEA grid around Vienna
AOI = {'west': 4780000, 'east': 4790000, 'south': 2830000, 'north': 2840000, 'crs': 3035}

### get processing_options for the eo_processing functions, collection_options and job_options

In [ ]:
processing_options = get_advanced_options(provider='cdse', skip_check_S1=False, skip_check_S2=True)
job_options = get_job_options(provider='cdse', task='feature_generation')
collection_options = get_collection_options(provider='cdse')
processing_options.update({'openeo_chunk_size': 64})

In [ ]:
processing_options

### establish connection to openEO

In [ ]:
con = init_connection(provider='cdse')

### run the feature cube generation WITHOUT NVBT band

In [ ]:
# update job_options due toBerts setting from last inference runs
# ToDO: optimize the job settings for feature_cube_generation_with_nobs & feature_cube_generation and put in settings + add correct task to 'get_job_options'
job_options.update({
    "driver-memory": "4G",
    "driver-memoryOverhead": "4G",
    "executor-memory": "5G",
    "executor-memoryOverhead": "3g",
    "max-executors": 10,
    "python-memory": "disable",
    "allow_empty_cubes": True,
    "soft-errors": 0.05})

In [ ]:
# get master cube without nobs
data = generate_master_feature_cube(con, AOI, start, end, **collection_options, **processing_options)

In [ ]:
data.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\features_cube_v5.tif', title='feature without nobs (10x10km)', job_options=job_options)

## run with activated NVBT generation

In [ ]:
# now we run same with nobs_perc band
processing_options.update({'get_NVBT': True})
data2 = generate_master_feature_cube(con, AOI, start, end, **collection_options, **processing_options)

In [ ]:
data2.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\features_cube_with_nobs_v5.tif', title='feature with nobs (10x10km)', job_options=job_options)